## Document Preprocessing

In [1]:
import fitz
from tqdm.auto import tqdm

pdf_path = "trial_paper.pdf"

def text_formatter(text: str) -> str:
    return text.replace("\n", " ").strip()

doc = fitz.open(pdf_path)
pages_and_texts = []

for page_number, page in tqdm(enumerate(doc)):
    text = page.get_text()
    text = text_formatter(text)
    
    pages_and_texts.append({
        "page_number": page_number,
        "page_char_count": len(text),
        "page_word_count": len(text.split(" ")),
        "text": text
    })

/Users/architsingal/Desktop/archit/python/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
35it [00:00, 129.11it/s]


## Sentence Splitting and Chunking

In [2]:
from spacy.lang.en import English
import pandas as pd
import re

nlp = English()
nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    item["sentences"] = [str(sentence) for sentence in nlp(item["text"]).sents]
    
chunk_size = 10
def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"], slice_size=chunk_size)

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) 
        
        chunk_dict["sentence_chunk"] = joined_sentence_chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # roughly 4 chars per token
        
        pages_and_chunks.append(chunk_dict)

min_token_length = 30
pages_and_chunks = [chunk for chunk in pages_and_chunks if chunk["chunk_token_count"] > min_token_length]

100%|██████████| 35/35 [00:00<00:00, 32988.91it/s]


## Embedding Creation

In [3]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=device)

text_chunks = [item["sentence_chunk"] for item in pages_and_chunks]

embeddings = embedding_model.encode(
    text_chunks,
    batch_size=32, 
    convert_to_tensor=True
)

for i, item in enumerate(pages_and_chunks):
    item["embedding"] = embeddings[i]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21532.00it/s]


## Retrieval

In [4]:
from sentence_transformers import util

def retrieve_relevant_resources(query: str, embeddings: torch.tensor, n_resources: int = 3):
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)
    
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    
    scores, indices = torch.topk(input=dot_scores, k=n_resources)
    
    return scores, indices

query = "What is rank one model editing and which modules does it modify?"
scores, indices = retrieve_relevant_resources(query, embeddings)

context_items = [pages_and_chunks[i] for i in indices]

## Augmentation and Generation

In [5]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY environment variable not found. Please set it before running.")

genai.configure(api_key=api_key)

llm_model = genai.GenerativeModel('gemini-2.5-flash')

def prompt_formatter(query: str, context_items: list[dict]) -> str:
    """Formats the retrieved context and query into a prompt."""
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])
    
    prompt = f"""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Make sure your answers are as explanatory as possible based ONLY on the context.

Context:
{context}

Query: {query}
Answer:"""
    
    return prompt

prompt = prompt_formatter(query, context_items)

outputs = llm_model.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        max_output_tokens=2048,
        temperature=0.7,
    )
)

final_answer = outputs.text

print(f"RAG Output:\n{final_answer}")

/var/folders/3r/5vblc6zn4sq9rbwl4rv64py80000gn/T/ipykernel_25594/2225766252.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


RAG Output:
**Relevant Passages:**

*   "E.5 [GPT-2 XL, GPT-J] Rank-One Model Editing (ROME) ROME’s update (Section 3.1) consists of key selection (Eqn.3), value optimization (Eqn.4), and v insertion (Appendix A). We perform the intervention at layer 18. As Figure 1k shows, this is the center of causal effect in MLP layers, and as Figure 3 shows, layer 18 is approximately when MLP outputs begin to switch from acting as keys to values."
*   "Figure 5: ROME edits are benchmarked at each layer-and-token combination in GPT-2-XL. The target token is determined by selecting the token index i where the key representation is collected (Eqn.3). ROME editing results conﬁrm the importance of mid-layer MLP layers at the ﬁnal subject token, where performance peaks."

**Answer:**

Rank-One Model Editing (ROME) is a model editing technique whose update process consists of three main steps: key selection, value optimization, and v insertion.

ROME performs its intervention at layer 18 of the model. Th